In [165]:
import pandas as pd


# ============================================================
# 1. FILE PATHS
# ============================================================

AOD_FILE = r"C:/Users/Juvair/Desktop/Job Application CV's/BUET_RA/Dataset/Air Pollution Dataset/AOD-14-21-daywise.csv"

CAMS_FILE = r"C:/Users/Juvair/Desktop/Job Application CV's/BUET_RA/Dataset/Air Pollution Dataset/CAMS_All_Stations_v2.csv"

OUTPUT_FILE = r"C:/Users/Juvair/Desktop/Job Application CV's/BUET_RA/Dataset/Air Pollution Dataset/AOD_CAMS_merged.csv"

OUTPUT_FILE_DateCorrection = r"C:/Users/Juvair/Desktop/Job Application CV's/BUET_RA/Dataset/Air Pollution Dataset/CAMS_All_Stations_v3.csv"



In [166]:
# ============================================================
# 2. READ DATA
# ============================================================

print("Reading AOD data...")

aod = pd.read_csv(AOD_FILE)

print("AOD data loaded.")
print("AOD shape:", aod.shape)
print("AOD columns:")
print(aod.columns.tolist())
print(aod.head())


Reading AOD data...
AOD data loaded.
AOD shape: (2922, 13)
AOD columns:
['Time', 'month', 'year', 'Chittagong_Agrabad', 'Khulna', 'Dhaka_DarusSalam', 'Savar', 'Dhaka_BARC', 'Narayanganj', 'Chittagong_Khulshi', 'Sylhet', 'Rajshahi', 'Barishal']
       Time  month  year  Chittagong_Agrabad    Khulna  Dhaka_DarusSalam  \
0  1/1/2014      1  2014            0.265612  0.667722          0.383040   
1  2/1/2014      1  2014            0.369412  0.897687          0.897816   
2  3/1/2014      1  2014            0.623535  1.358403          1.012017   
3  4/1/2014      1  2014            0.582953  0.875460          0.968586   
4  5/1/2014      1  2014            0.593965  1.507097          0.999368   

      Savar  Dhaka_BARC  Narayanganj  Chittagong_Khulshi    Sylhet  Rajshahi  \
0  0.474722    0.370550     0.432322            0.286383  0.291381  0.471159   
1  0.913665    0.879233     0.891694            0.384244  0.569352  0.987994   
2  1.034903    1.046894     1.123156            0.651661  0

In [167]:
# ============================================================
# 3. CONVERT AOD DATE
# ============================================================

# Your AOD Time column contains mixed formats:
#
# 1/1/2014
# 2/1/2014
# ...
# 12/1/2014
# 13-01-2014
# 14-01-2014
#
# dayfirst=True handles both formats correctly.

aod["Time"] = pd.to_datetime(
    aod["Time"],
    dayfirst=True,
    errors="coerce"
)


# Check if any dates could not be converted

invalid_aod_dates = aod["Time"].isna().sum()

print("Invalid AOD dates:", invalid_aod_dates)


Invalid AOD dates: 1770


In [168]:
# ============================================================
# 4. CONVERT AOD FROM WIDE FORMAT TO LONG FORMAT
# ============================================================

# Current format:
#
# Time       Agrabad    Baira    Darus Salam ...
# 2014-01-01  0.26      0.66     0.38
#
# We convert it to:
#
# Time        Station       AOD
# 2014-01-01  Agrabad       0.26
# 2014-01-01  Baira         0.66
# 2014-01-01  Darus Salam   0.38

aod_long = aod.melt(
    id_vars=["Time", "month", "year"],
    var_name="Station",
    value_name="AOD"
)


# Remove rows where AOD is missing

aod_long = aod_long.dropna(subset=["AOD"])


print("\nAOD long-format shape:", aod_long.shape)

print("\nAOD long-format sample:")
print(aod_long.head())


AOD long-format shape: (16424, 5)

AOD long-format sample:
        Time  month  year             Station       AOD
0 2014-01-01      1  2014  Chittagong_Agrabad  0.265612
1 2014-01-02      1  2014  Chittagong_Agrabad  0.369412
2 2014-01-03      1  2014  Chittagong_Agrabad  0.623535
3 2014-01-04      1  2014  Chittagong_Agrabad  0.582953
4 2014-01-05      1  2014  Chittagong_Agrabad  0.593965


In [169]:

# ============================================================
# 5. READ CAMS DATA
# ============================================================

print("\nReading CAMS data...")

cams = pd.read_csv(CAMS_FILE)

print("CAMS data loaded.")
print("CAMS shape:", cams.shape)

print("\nCAMS columns:")
print(cams.columns.tolist())

#print(cams.head())


Reading CAMS data...
CAMS data loaded.
CAMS shape: (17449, 22)

CAMS columns:
['CalendarDate', 'Date', 'CAMS_Station', 'SO2_daily_avg', 'NO_daily_avg', 'NO2_daily_avg', 'NOX_daily_avg', 'CO_daily_avg', 'CO8hr_daily_avg', 'O3_daily_avg', 'O38hr_daily_avg', 'PM25_daily_avg', 'PM10_daily_avg', 'WindSpeed_daily_avg', 'Temp_daily_avg', 'RH_daily_avg', 'SolarRad_daily_avg', 'BP_daily_avg', 'Rain_daily_total', 'VWindSpeed_daily_avg', 'hours_available', 'WindDir_daily_circular_mean']


In [170]:
# ============================================================
# 6. CONVERT CAMS DATE
# ============================================================

cams["CalendarDate"] = pd.to_datetime(
    cams["CalendarDate"],
    dayfirst=True,
    errors="coerce"
)


# Check invalid CAMS dates

invalid_cams_dates = cams["CalendarDate"].isna().sum()

print("Invalid CAMS dates:", invalid_cams_dates)

Invalid CAMS dates: 10669


In [171]:
# ============================================================
# CHECK CAMS DATE FORMAT
# ============================================================

print(cams["CalendarDate"].head(20))
print("\nData type:")
print(cams["CalendarDate"].dtype)

print("\nSome unique date values:")
print(cams["CalendarDate"].dropna().astype(str).head(20).tolist())

0    2012-01-11
1    2012-02-11
2    2012-03-11
3    2012-04-11
4    2012-05-11
5    2012-06-11
6    2012-07-11
7    2012-08-11
8    2012-09-11
9    2012-10-11
10   2012-11-11
11   2012-12-11
12          NaT
13          NaT
14          NaT
15          NaT
16          NaT
17          NaT
18          NaT
19          NaT
Name: CalendarDate, dtype: datetime64[ns]

Data type:
datetime64[ns]

Some unique date values:
['2012-01-11', '2012-02-11', '2012-03-11', '2012-04-11', '2012-05-11', '2012-06-11', '2012-07-11', '2012-08-11', '2012-09-11', '2012-10-11', '2012-11-11', '2012-12-11', '2012-02-12', '2012-04-12', '2012-05-12', '2012-06-12', '2012-07-12', '2012-08-12', '2012-09-12', '2012-10-12']


In [60]:
# # ============================================================
# # Date Field Correction
# # ============================================================
# import pandas as pd

# df = pd.read_excel(CAMS_FILE)

# print("Original data:")
# print(df.head())

# print("\nOriginal CalendarDate data type:")
# print(df["CalendarDate"].dtype)


# # ============================================================
# #CONVERT CalendarDate TO STRING
# # ============================================================

# df["CalendarDate"] = df["CalendarDate"].astype(str).str.strip()


# # ============================================================
# #  CONVERT MIXED DATE FORMATS TO DATETIME
# # ============================================================

# # Handles both:
# # 11/1/2012
# # 2012-11-02
# # 11/7/2012
# # 2012-11-09
# #
# # format='mixed' allows Pandas to interpret different
# # date formats within the same column.

# df["Date"] = pd.to_datetime(
#     df["CalendarDate"],
#     format="mixed",
#     errors="coerce"
# )

# # ============================================================
# #  CHECK INVALID DATES
# # ============================================================

# invalid_dates = df["Date"].isna().sum()

# print("\nInvalid dates:", invalid_dates)


# # Show invalid original values, if any

# if invalid_dates > 0:

#     print("\nInvalid CalendarDate values:")

#     print(
#         df.loc[
#             df["Date"].isna(),
#             "CalendarDate"
#         ].head(20)
#     )
# # ============================================================
# #  CHECK RESULT
# # ============================================================

# print("\nConverted dates:")

# print(
#     df[
#         ["CalendarDate", "Date"]
#     ].head(20)
# )

# # ============================================================
# # SAVE
# # ============================================================

# df.to_csv(
#     OUTPUT_FILE_DateCorrection,
#     index=False
# )

# print("\n========================================")
# print("DONE")
# print("========================================")

# print("Output file:")
# print(OUTPUT_FILE)



Original data:
  CalendarDate        CAMS_Station  SO2_daily_avg  NO_daily_avg  \
0   2012-11-01  Chittagong_Agrabad       0.387143      3.818824   
1   2012-11-02  Chittagong_Agrabad       0.140000     12.406429   
2   2012-11-03  Chittagong_Agrabad       0.056667      2.007826   
3   2012-11-04  Chittagong_Agrabad       0.035000      4.373750   
4   2012-11-05  Chittagong_Agrabad       0.292857     12.615652   

   NO2_daily_avg  NOX_daily_avg  CO_daily_avg  CO8hr_daily_avg  O3_daily_avg  \
0       4.949167       7.934783      0.607083         0.549191      2.723478   
1       4.708000      15.477647      0.664783         0.684583      3.462632   
2       5.911667       7.870417      0.507917         0.514635      3.591667   
3       5.375000      10.089545      0.348000         0.371029      1.157500   
4       5.739524      20.112500      0.655000         0.465298      1.718000   

   O38hr_daily_avg  ...  PM10_daily_avg  WindSpeed_daily_avg  Temp_daily_avg  \
0         5.046912  .

In [174]:
# ============================================================
# 7. RENAME AOD STATIONS TO MATCH CAMS STATIONS
# ============================================================

station_mapping = {
    "Agrabad": "Chittagong_Agrabad",
    "Baira": "Khulna",
    "Darus Salam": "Dhaka_DarusSalam",
    "East Chandana": "Savar",
    "Farmgate": "Dhaka_BARC",
    "Khanpur": "Narayanganj",
    "Khulshi": "Chittagong_Khulshi",
    "Red Crescent Office": "Sylhet",
    "Sopura": "Rajshahi",
    "Uttar Bagura Road": "Barishal"
}

aod_long["Station"] = aod_long["Station"].replace(station_mapping)

# Remove extra spaces
aod_long["Station"] = aod_long["Station"].str.strip()
cams["CAMS_Station"] = cams["CAMS_Station"].str.strip()

print(aod_long.head())
print("\n",cams.head())


        Time  month  year             Station       AOD
0 2014-01-01      1  2014  Chittagong_Agrabad  0.265612
1 2014-01-02      1  2014  Chittagong_Agrabad  0.369412
2 2014-01-03      1  2014  Chittagong_Agrabad  0.623535
3 2014-01-04      1  2014  Chittagong_Agrabad  0.582953
4 2014-01-05      1  2014  Chittagong_Agrabad  0.593965

   CalendarDate       Date        CAMS_Station  SO2_daily_avg  NO_daily_avg  \
0   2012-01-11  11/1/2012  Chittagong_Agrabad       0.387143      3.818824   
1   2012-02-11  11/2/2012  Chittagong_Agrabad       0.140000     12.406429   
2   2012-03-11  11/3/2012  Chittagong_Agrabad       0.056667      2.007826   
3   2012-04-11  11/4/2012  Chittagong_Agrabad       0.035000      4.373750   
4   2012-05-11  11/5/2012  Chittagong_Agrabad       0.292857     12.615652   

   NO2_daily_avg  NOX_daily_avg  CO_daily_avg  CO8hr_daily_avg  O3_daily_avg  \
0       4.949167       7.934783      0.607083         0.549191      2.723478   
1       4.708000      15.477647  

In [175]:
# ============================================================
# 8. CHECK STATION NAMES
# ============================================================

print("\nAOD stations:")
print(sorted(aod_long["Station"].unique()))

print("\nCAMS stations:")
print(sorted(cams["CAMS_Station"].unique()))


AOD stations:
['Barishal', 'Chittagong_Agrabad', 'Chittagong_Khulshi', 'Dhaka_BARC', 'Dhaka_DarusSalam', 'Khulna', 'Narayanganj', 'Rajshahi', 'Savar', 'Sylhet']

CAMS stations:
['Chittagong_Agrabad', 'Chittagong_Khulshi', 'Cumilla', 'Dhaka_BARC', 'Dhaka_DarusSalam', 'Gazipur', 'Khulna', 'Mymensingh', 'Narayanganj', 'Narsingdi', 'Rajshahi', 'Rangpur', 'Savar']


In [176]:
print("AOD:")
print(aod_long[["Station", "Time"]].head())
print(aod_long["Time"].dtype)

print("\nCAMS:")
print(cams[["CAMS_Station", "Date"]].head())
print(cams["Date"].dtype)

AOD:
              Station       Time
0  Chittagong_Agrabad 2014-01-01
1  Chittagong_Agrabad 2014-01-02
2  Chittagong_Agrabad 2014-01-03
3  Chittagong_Agrabad 2014-01-04
4  Chittagong_Agrabad 2014-01-05
datetime64[ns]

CAMS:
         CAMS_Station       Date
0  Chittagong_Agrabad  11/1/2012
1  Chittagong_Agrabad  11/2/2012
2  Chittagong_Agrabad  11/3/2012
3  Chittagong_Agrabad  11/4/2012
4  Chittagong_Agrabad  11/5/2012
object


In [177]:
aod_long["Date"] = pd.to_datetime(
    aod_long["Time"],
    format="mixed",
    errors="coerce"
).dt.normalize()

cams["Date"] = pd.to_datetime(
    cams["Date"],
    format="mixed",
    errors="coerce"
).dt.normalize()

print(aod_long.head())
print(aod_long["Time"].dtype)
print("\n")
print(cams.head())
print(cams["Date"].dtype)

        Time  month  year             Station       AOD       Date
0 2014-01-01      1  2014  Chittagong_Agrabad  0.265612 2014-01-01
1 2014-01-02      1  2014  Chittagong_Agrabad  0.369412 2014-01-02
2 2014-01-03      1  2014  Chittagong_Agrabad  0.623535 2014-01-03
3 2014-01-04      1  2014  Chittagong_Agrabad  0.582953 2014-01-04
4 2014-01-05      1  2014  Chittagong_Agrabad  0.593965 2014-01-05
datetime64[ns]


  CalendarDate       Date        CAMS_Station  SO2_daily_avg  NO_daily_avg  \
0   2012-01-11 2012-11-01  Chittagong_Agrabad       0.387143      3.818824   
1   2012-02-11 2012-11-02  Chittagong_Agrabad       0.140000     12.406429   
2   2012-03-11 2012-11-03  Chittagong_Agrabad       0.056667      2.007826   
3   2012-04-11 2012-11-04  Chittagong_Agrabad       0.035000      4.373750   
4   2012-05-11 2012-11-05  Chittagong_Agrabad       0.292857     12.615652   

   NO2_daily_avg  NOX_daily_avg  CO_daily_avg  CO8hr_daily_avg  O3_daily_avg  \
0       4.949167       7.934783 

In [178]:
aod_long["Station"] = aod_long["Station"].astype(str).str.strip()

#print("aod_long dataset\n",aod_long)

cams["CAMS_Station"] = cams["CAMS_Station"].astype(str).str.strip()

#print("\n cams dataset \n",cams)

In [185]:
# import pandas as pd

# # 1. Ensure date columns are in datetime format
# aod_long['Date'] = pd.to_datetime(aod_long['Date'])
# cams['Date'] = pd.to_datetime(cams['Date'])

# # 2. Standardize station names to avoid missing matches due to whitespaces
# aod_long['Station'] = aod_long['Station'].astype(str).str.strip()
# cams['CAMS_Station'] = cams['CAMS_Station'].astype(str).str.strip()

# 3. Merge datasets using Date and Station as keys
merged_df = pd.merge(
    aod_long,
    cams,
    left_on=['Date', 'Station'],
    right_on=['Date', 'CAMS_Station'],
    how='inner',  # Use 'left' or 'outer' if you want to keep unmatched rows
)

print(merged_df.head())

# 4. (Optional) Drop the redundant CAMS_Station column
merged_df = merged_df.drop(columns=['CAMS_Station'])

# Save the DataFrame to CSV
merged_df.to_csv(OUTPUT_FILE, index=False)

print("\n \nFile successfully saved!")

        Time  month  year             Station       AOD       Date  \
0 2014-01-07      1  2014  Chittagong_Agrabad  0.464947 2014-01-07   
1 2014-01-08      1  2014  Chittagong_Agrabad  0.581518 2014-01-08   
2 2014-01-09      1  2014  Chittagong_Agrabad  0.321300 2014-01-09   
3 2014-01-12      1  2014  Chittagong_Agrabad  1.173271 2014-01-12   
4 2014-02-01      2  2014  Chittagong_Agrabad  0.275000 2014-02-01   

  CalendarDate        CAMS_Station  SO2_daily_avg  NO_daily_avg  ...  \
0   2014-07-01  Chittagong_Agrabad       4.845909     76.835714  ...   
1   2014-08-01  Chittagong_Agrabad       4.733333      0.145714  ...   
2   2014-09-01  Chittagong_Agrabad       4.481250      0.288182  ...   
3   2014-12-01  Chittagong_Agrabad       8.054737     37.443077  ...   
4   2014-01-02  Chittagong_Agrabad       6.243333      0.144286  ...   

   PM10_daily_avg  WindSpeed_daily_avg  Temp_daily_avg  RH_daily_avg  \
0      245.814286             2.811818       19.140909     67.089091   
1 